# Análise de Dados — Inventário SBF
Tratamento complementar, colunas calculadas e preparação da base para o Dashboard.

In [12]:
import pandas as pd

df = pd.read_excel("dados-tratados.xlsx")
print(f"Base carregada: {len(df)} produtos, {len(df.columns)} colunas")
print(f"Colunas: {list(df.columns)}")
df.head()

Base carregada: 100 produtos, 14 colunas
Colunas: ['ID_Produto', 'Nome_Produto', 'Categoria', 'Data_Registro', 'Estoque_Atual', 'Estoque_Minimo', 'Custo_Unitario', 'Venda_Media_Diaria', 'Lead_Time_Fornecedor', 'Status_Entrega', 'Valor_Total_Estoque', 'Cobertura_Estoque_Dias', 'Top_20%_Valor', 'Data_Prevista_Chegada']


,ID_Produto,Nome_Produto,Categoria,Data_Registro,Estoque_Atual,Estoque_Minimo,Custo_Unitario,Venda_Media_Diaria,Lead_Time_Fornecedor,Status_Entrega,Valor_Total_Estoque,Cobertura_Estoque_Dias,Top_20%_Valor,Data_Prevista_Chegada
0,P001,Tenis Nike Air Max,Calçados,2026-01-15,45,20,299.9,5,7,No Prazo,13495.5,9.0,Não,2026-01-22
1,P002,Camiseta Dry Fit,Roupas,NaT,120,50,59.9,12,3,No Prazo,7188.0,10.0,Não,NaT
2,P003,Bola de Futebol,Acessórios,NaT,10,15,89.0,4,5,Atrasado,890.0,2.5,Não,NaT
3,P004,Shorts Corrida,Roupas,2026-01-18,0,30,45.0,8,4,No Prazo,0.0,0.0,Não,2026-01-22
4,P005,Tenis Centauro Run,Calçados,NaT,0,25,150.0,10,10,Atrasado,0.0,0.0,Não,NaT


## Colunas Calculadas para as Perguntas

In [13]:
# P2: Valor Total em Estoque
if "Valor_Total_Estoque" not in df.columns:
    df["Valor_Total_Estoque"] = df["Estoque_Atual"] * df["Custo_Unitario"]

# P3: Cobertura de Estoque (dias)
df["Cobertura_Estoque_Dias"] = (df["Estoque_Atual"] / df["Venda_Media_Diaria"]).round(1)

# P4: Flag Top 20% mais caros (Pareto)
df_sorted = df.sort_values("Valor_Total_Estoque", ascending=False)
df_sorted["VTE_Acumulado"] = df_sorted["Valor_Total_Estoque"].cumsum()
limite_20 = df_sorted["Valor_Total_Estoque"].sum() * 0.20
ids_top20 = df_sorted[df_sorted["VTE_Acumulado"] <= limite_20]["ID_Produto"].tolist()
df["Top_20_Valor"] = df["ID_Produto"].isin(ids_top20).map({True: "Sim", False: "Não"})

# P5: Data Prevista de Chegada
df["Data_Prevista_Chegada"] = df["Data_Registro"] + pd.to_timedelta(df["Lead_Time_Fornecedor"], unit="D")

# Salvar base completa
df.to_excel("dados-tratados.xlsx", index=False)

print("=" * 55)
print("BASE PREPARADA PARA O DASHBOARD")
print("=" * 55)
print(f"Colunas adicionadas:")
print(f"  • Valor_Total_Estoque      (P2)")
print(f"  • Cobertura_Estoque_Dias   (P3)")
print(f"  • Top_20_Valor             (P4) → {len(ids_top20)} produtos")
print(f"  • Data_Prevista_Chegada    (P5)")

chegada_apos = df[df["Data_Prevista_Chegada"] > pd.Timestamp("2026-02-15")]
print(f"\n  P5: Produtos com chegada após 15/02/2026: {len(chegada_apos)}")
print(f"\nTotal colunas: {len(df.columns)}")
df.head()

BASE PREPARADA PARA O DASHBOARD
Colunas adicionadas:
  • Valor_Total_Estoque      (P2)
  • Cobertura_Estoque_Dias   (P3)
  • Top_20_Valor             (P4) → 5 produtos
  • Data_Prevista_Chegada    (P5)

  P5: Produtos com chegada após 15/02/2026: 26

Total colunas: 15


,ID_Produto,Nome_Produto,Categoria,Data_Registro,Estoque_Atual,Estoque_Minimo,Custo_Unitario,Venda_Media_Diaria,Lead_Time_Fornecedor,Status_Entrega,Valor_Total_Estoque,Cobertura_Estoque_Dias,Top_20%_Valor,Data_Prevista_Chegada,Top_20_Valor
0,P001,Tenis Nike Air Max,Calçados,2026-01-15,45,20,299.9,5,7,No Prazo,13495.5,9.0,Não,2026-01-22,Não
1,P002,Camiseta Dry Fit,Roupas,NaT,120,50,59.9,12,3,No Prazo,7188.0,10.0,Não,NaT,Não
2,P003,Bola de Futebol,Acessórios,NaT,10,15,89.0,4,5,Atrasado,890.0,2.5,Não,NaT,Não
3,P004,Shorts Corrida,Roupas,2026-01-18,0,30,45.0,8,4,No Prazo,0.0,0.0,Não,2026-01-22,Não
4,P005,Tenis Centauro Run,Calçados,NaT,0,25,150.0,10,10,Atrasado,0.0,0.0,Não,NaT,Não
